# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aijaz-khalique/flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



Unit of analysis: One row represents one content item for one client.

The raw warehouse contains daily client-content performance records. I aggregate these records to one row per client-content pair.

Time window:
- February 2026 is the feature window.
- March 2026 is the future outcome window.
- The decision point is 2026-02-28.

Prediction target: `went_dark`, where 1 means the content recorded zero measured GSC clicks during March 2026 and 0 means it recorded at least one measured GSC click.

I keep the feature and outcome windows separate so that future information is not used to create the features.

In [9]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

In [10]:
from dotenv import load_dotenv

load_dotenv(override=True)

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    hf_token = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

print("Hugging Face token loaded:", hf_token is not None)

Hugging Face token loaded: True


In [11]:
import requests

response = requests.get(
    "https://huggingface.co/api/whoami-v2",
    headers={"Authorization": f"Bearer {hf_token}"}
)

print("Status:", response.status_code)

if response.status_code == 200:
    print("Hugging Face authentication successful.")
else:
    print("Authentication failed.")
    print(response.text[:500])

Status: 200
Hugging Face authentication successful.


In [12]:
con = duckdb.connect()

con.execute(
    "SET VARIABLE hf_token = ?",
    [hf_token]
)

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"

print("DuckDB connected.")
print("February feature window ready.")
print("March label window ready.")

DuckDB connected.
February feature window ready.
March label window ready.


In [13]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {FEB}
""").df()

,total_rows
0,7355108


In [14]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {MAR}
""").df()

,total_rows
0,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Features

The features are calculated from information available during February 2026:

- `imp_feb`
- `clk_feb`
- `ctr_feb`
- `pos_feb`
- `pos_volatility_feb`
- `days_with_imps_feb`
- `zero_click_days_feb`

### Label

- `went_dark`

The label is calculated from March 2026 GSC clicks.

### Context

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `gsc_data_available`
- `is_published`
- `is_deleted`

These fields are useful for joining, filtering, grouping, and reporting.

### Excluded

`client_hash_id` and `content_hash_id` are excluded from model features because they are identifiers and could allow the model to memorize specific clients or content.

March performance fields are also excluded from the features because March is the future outcome window.

In [15]:
con.sql(f"""
SELECT
    'February 2026' AS window,
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}

UNION ALL

SELECT
    'March 2026' AS window,
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
""").df()

,window,total_rows,first_date,last_date
0,February 2026,7355108,2026-02-01,2026-02-28
1,March 2026,9841378,2026-03-01,2026-03-31


In [16]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS duplicate_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {FEB}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").df()

grain_check

,duplicate_groups
0,0


In [17]:
feb_agg = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS imp_feb,

    SUM(gsc_clicks) AS clk_feb,

    SUM(gsc_clicks) / NULLIF(
        SUM(gsc_impressions), 0
    ) AS ctr_feb,

    SUM(gsc_sum_position) / NULLIF(
        SUM(gsc_impressions), 0
    ) AS pos_feb,

    STDDEV(gsc_avg_position) AS pos_volatility_feb,

    COUNT(*) FILTER (
        WHERE gsc_impressions > 0
    ) AS days_with_imps_feb,

    COUNT(*) FILTER (
        WHERE gsc_impressions > 0
        AND gsc_clicks = 0
    ) AS zero_click_days_feb

FROM {FEB}

WHERE gsc_data_available = TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    SUM(gsc_impressions) >= 100
    AND SUM(gsc_clicks) >= 3
""").df()

print("February feature rows:", len(feb_agg))

feb_agg.head()

February feature rows: 29729


,client_hash_id,content_hash_id,imp_feb,clk_feb,ctr_feb,pos_feb,pos_volatility_feb,days_with_imps_feb,zero_click_days_feb
0,client_fef1a8f436438636,content_0c76102b8fd919a3,836.0,3.0,0.003589,4.758373,5.267814,28,26
1,client_fef1a8f436438636,content_f3cd95d50a26b509,6488.0,4.0,0.000617,7.127928,1.840281,28,25
2,client_fef1a8f436438636,content_322e2fb30acbe0bc,1567.0,4.0,0.002553,3.415444,1.545334,28,24
3,client_fef1a8f436438636,content_7a02e4bb9d4a379a,23187.0,148.0,0.006383,3.556260,0.571481,28,0
4,client_fef1a8f436438636,content_14754ac78932ab6a,7601.0,34.0,0.004473,11.287857,1.854103,28,7


In [18]:
content_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    content_type,
    word_count,
    category_count,
    search_volume,
    competition,
    competition_level,
    cpc,
    is_published,
    is_deleted,
    content_created_date

FROM {DIM}

WHERE is_published = TRUE
""").df()

print("Content rows:", len(content_features))

content_features.head()

Content rows: 411540


,client_hash_id,content_hash_id,content_type,word_count,category_count,search_volume,competition,competition_level,cpc,is_published,is_deleted,content_created_date
0,client_04660893ae39614a,content_004de9653278b5a4,keyword article,2555,3,30,0.91,HIGH,0.98,True,False,2026-05-30
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword article,2430,4,10,0.00,LOW,0.00,True,False,2026-06-12
2,client_04660893ae39614a,content_01410f2556c327ac,keyword article,2645,4,480,0.36,MEDIUM,0.62,True,False,2026-05-09
3,client_04660893ae39614a,content_019f27f634053ca7,keyword article,2522,4,0,0.00,LOW,0.00,True,False,2026-06-15
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword article,2552,4,2400,0.70,HIGH,0.90,True,False,2026-05-21


In [19]:
feature_frame = feb_agg.merge(
    content_features,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("Feature frame rows:", len(feature_frame))

feature_frame.head()

Feature frame rows: 29700


,client_hash_id,content_hash_id,imp_feb,clk_feb,ctr_feb,pos_feb,pos_volatility_feb,days_with_imps_feb,zero_click_days_feb,content_type,word_count,category_count,search_volume,competition,competition_level,cpc,is_published,is_deleted,content_created_date
0,client_fef1a8f436438636,content_0c76102b8fd919a3,836.0,3.0,0.003589,4.758373,5.267814,28,26,keyword article,2981,0,30,1.00,HIGH,1.79,True,False,2025-07-11
1,client_fef1a8f436438636,content_f3cd95d50a26b509,6488.0,4.0,0.000617,7.127928,1.840281,28,25,keyword article,2763,0,50,0.02,LOW,0.00,True,False,2025-07-11
2,client_fef1a8f436438636,content_322e2fb30acbe0bc,1567.0,4.0,0.002553,3.415444,1.545334,28,24,keyword article,1525,0,10,0.04,LOW,0.00,True,False,2025-07-11
3,client_fef1a8f436438636,content_7a02e4bb9d4a379a,23187.0,148.0,0.006383,3.556260,0.571481,28,0,keyword article,2057,0,70,0.00,LOW,0.00,True,False,2025-07-11
4,client_fef1a8f436438636,content_14754ac78932ab6a,7601.0,34.0,0.004473,11.287857,1.854103,28,7,keyword article,2849,0,20,0.03,LOW,0.30,True,False,2025-07-11


In [20]:
duplicates = feature_frame.duplicated(
    subset=[
        "client_hash_id",
        "content_hash_id"
    ]
).sum()

print("Duplicate client-content rows:", duplicates)

assert duplicates == 0

Duplicate client-content rows: 0


In [21]:
march_label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS imp_mar,

    SUM(gsc_clicks) AS clk_mar,

    COUNT(*) FILTER (
        WHERE gsc_data_available = TRUE
    ) AS measured_days_mar

FROM {MAR}

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

march_label.head()

,client_hash_id,content_hash_id,imp_mar,clk_mar,measured_days_mar
0,client_62f4a7e64f5e0096,content_26b2a4161a2fd685,611.0,2.0,31
1,client_62f4a7e64f5e0096,content_e7d8805c11d72d98,10113.0,35.0,31
2,client_62f4a7e64f5e0096,content_37a6fac676c8cebb,48049.0,4.0,31
3,client_62f4a7e64f5e0096,content_7647b83ab130cb06,1830.0,29.0,31
4,client_62f4a7e64f5e0096,content_dacd2734a5b82e0e,1476.0,3.0,31


In [22]:
frame = feature_frame.merge(
    march_label,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="left"
)

print("Rows before label filtering:", len(frame))

Rows before label filtering: 29700


In [23]:
frame = frame[
    frame["measured_days_mar"].fillna(0) > 0
].copy()

frame["imp_mar"] = frame["imp_mar"].fillna(0)

frame["clk_mar"] = frame["clk_mar"].fillna(0)

frame["went_dark"] = (
    frame["clk_mar"] == 0
).astype(int)

print("Final rows:", len(frame))

print("\nLabel distribution:")
print(frame["went_dark"].value_counts())

Final rows: 29353

Label distribution:
went_dark
0    28194
1     1159
Name: count, dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


The checks above verify that the raw data is daily at the client-content-date level and that the feature frame is aggregated to one row per client-content pair.

February 2026 is used for features and March 2026 is used only for the future outcome.

The February universe requires at least 100 impressions and 3 clicks so that very small amounts of search activity are not treated as a strong baseline.

March rows without measured GSC data are not automatically treated as zero clicks. This avoids confusing missing data with a true zero-click outcome.

In [24]:
model_features = [
    "imp_feb",
    "clk_feb",
    "ctr_feb",
    "pos_feb",
    "pos_volatility_feb",
    "days_with_imps_feb",
    "zero_click_days_feb"
]

missing_check = (
    frame[
        model_features + ["went_dark"]
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_check

pos_volatility_feb     2
imp_feb                0
ctr_feb                0
clk_feb                0
pos_feb                0
days_with_imps_feb     0
zero_click_days_feb    0
went_dark              0
dtype: int64

In [25]:
model_frame = frame[
    model_features + ["went_dark"]
].dropna().copy()

print("Final model rows:", len(model_frame))

print("\nLabel distribution:")
print(model_frame["went_dark"].value_counts())

Final model rows: 29351

Label distribution:
went_dark
0    28192
1     1159
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



This data cannot prove why a content item went dark.

The label shows that the content recorded zero measured GSC clicks in March, but it does not explain the cause. Possible causes include ranking changes, seasonality, changes in search intent, SERP changes, technical problems, or content quality.

GSC availability can vary across the data, so unavailable measurements should not automatically be interpreted as zero traffic.

The history may also be unbalanced across clients and content items.

The feature and label windows must remain separate. Using March information to create February features would cause target leakage and produce an unrealistic evaluation.

Therefore, this data is useful for prediction and decision support, but it does not provide causal evidence.

In [27]:
print("=" * 60)
print("ML-04 FINAL CHECK")
print("=" * 60)

duplicates = frame.duplicated(
    subset=[
        "client_hash_id",
        "content_hash_id"
    ]
).sum()

print("Final rows:", len(model_frame))
print("Number of features:", len(model_features))
print("Features:", model_features)

print("\nDuplicate client-content rows:", duplicates)

print("\nMissing values:")
print(model_frame.isna().sum())

print("\nLabel distribution:")
print(model_frame["went_dark"].value_counts())

assert duplicates == 0

assert len(model_features) == 7

assert "client_hash_id" not in model_features
assert "content_hash_id" not in model_features

assert "went_dark" not in model_features

assert model_frame.isna().sum().sum() == 0

print("\n✓ Data loaded")
print("✓ February feature window verified")
print("✓ March label window verified")
print("✓ Raw grain verified")
print("✓ Final grain verified")
print("✓ Missing values checked")
print("✓ Identifiers excluded")
print("✓ Future label excluded from features")
print("✓ ML-04 completed successfully")

ML-04 FINAL CHECK
Final rows: 29351
Number of features: 7
Features: ['imp_feb', 'clk_feb', 'ctr_feb', 'pos_feb', 'pos_volatility_feb', 'days_with_imps_feb', 'zero_click_days_feb']

Duplicate client-content rows: 0

Missing values:
imp_feb                0
clk_feb                0
ctr_feb                0
pos_feb                0
pos_volatility_feb     0
days_with_imps_feb     0
zero_click_days_feb    0
went_dark              0
dtype: int64

Label distribution:
went_dark
0    28192
1     1159
Name: count, dtype: int64

✓ Data loaded
✓ February feature window verified
✓ March label window verified
✓ Raw grain verified
✓ Final grain verified
✓ Missing values checked
✓ Identifiers excluded
✓ Future label excluded from features
✓ ML-04 completed successfully


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.